# Compliance analysis workflow

Ce notebook sert a piloter le pipeline et a interroger les resultats. La logique metier reste dans `src/compliance_nlp`.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

repo_root

In [ ]:
from compliance_nlp import (
    analyze_directory,
    load_generic_detection_rules,
    load_results,
    results_to_dataframe,
    summarize_results,
)

import pandas as pd

In [ ]:
input_dir = repo_root / "data" / "raw"
results_path = repo_root / "outputs" / "analysis" / "results.json"
generic_rules_path = repo_root / "configs" / "generic_detection_rules.csv"

input_dir, results_path, generic_rules_path

## Regles chargees

Cette cellule permet de verifier les regles centrales et leur niveau d'alerte.

In [ ]:
rules = load_generic_detection_rules(generic_rules_path)
rules_df = pd.DataFrame([
    {
        "rule_id": rule.rule_id,
        "rule_scope": rule.rule_scope,
        "category": rule.category,
        "terms": " | ".join(rule.terms),
        "alert_level": rule.alert_level,
    }
    for rule in rules
])
rules_df

## Relancer le pipeline si besoin

Laisse `RUN_PIPELINE = False` si tu veux seulement relire un resultat deja genere.

In [ ]:
RUN_PIPELINE = False

if RUN_PIPELINE:
    results = analyze_directory(input_dir, output_path=results_path)
else:
    results = load_results(results_path)

summary = summarize_results(results)
summary

In [ ]:
df = results_to_dataframe(results)
df

## Vues metier utiles

In [ ]:
df[["document_name", "code", "section", "matched_term", "alert_level", "severity"]]

In [ ]:
df[df["section"] == "beneficiaires"][
    ["document_name", "matched_term", "alert_level", "code", "evidence"]
].sort_values(["document_name", "matched_term"], na_position="last")

In [ ]:
df[df["section"] == "conseil"][
    ["document_name", "matched_term", "alert_level", "code", "evidence"]
].sort_values(["document_name", "matched_term"], na_position="last")

In [ ]:
df[df["alert_level"] == "interdit"][
    ["document_name", "section", "matched_term", "code", "evidence"]
].sort_values(["document_name", "matched_term"], na_position="last")

In [ ]:
df[df["matched_term"].notna()][
    ["document_name", "section", "matched_term", "alert_level"]
].sort_values(["matched_term", "document_name"])

In [ ]:
df.groupby(["section", "alert_level"], dropna=False).size().reset_index(name="count")

In [ ]:
df[df["code"] == "beneficiary_clause_imprecise"]

In [ ]:
df[df["code"] == "advice_above_financial_capacity"]